# Neural feature × ranking-loss ablation

Run the complete 2 × 2 neural ranking design on the shared article splits:

| Inputs | Pairwise logistic | Lambda nDCG |
|---|---|---|
| Metadata only | `metadata_pairwise` | `metadata_lambda` |
| Frozen BGE-M3 + metadata | `frozen_bge_pairwise` | `frozen_bge_lambda` |

Every variant writes to its own directory. The existing frozen-BGE and metadata-MLP notebook outputs are not modified. Lambda variants include one current highest-scoring negative per positive so each query has a non-zero top-k swap weight.

In [ ]:
import os
from dataclasses import asdict, replace
from pathlib import Path

from commentgap_analysis.neural_ranking import default_recipe, run_neural_ranker_workflow

MODEL_DATA_ROOT = Path(os.getenv('COMMENTGAP_MODEL_DATA_ROOT', 'model_output/selection_2025/model_data'))
DATA_ROOT = Path(os.getenv('COMMENTGAP_DATA_ROOT', 'data/scrape_2025'))
EMBEDDING_ROOT = Path(os.getenv('COMMENTGAP_EMBEDDING_ROOT', 'model_output/selection_2025/embeddings'))
OUTPUT_ROOT = Path(os.getenv('COMMENTGAP_NEURAL_ABLATION_ROOT', 'model_output/selection_2025/neural_rankers/feature_loss_ablation'))
TRAINING_MODE = os.getenv('COMMENTGAP_TRAINING_MODE', 'cv')
DEVICE = os.getenv('COMMENTGAP_DEVICE', 'auto')
BOOTSTRAP_DRAWS = int(os.getenv('COMMENTGAP_BOOTSTRAP_DRAWS', '1000'))
PROGRESS_EVERY = int(os.getenv('COMMENTGAP_NEURAL_PROGRESS_EVERY_STORIES', '100'))
FORCE_RECOMPUTE = os.getenv('COMMENTGAP_FORCE_RECOMPUTE', '0').lower() in {'1', 'true', 'yes'}

metadata_base = default_recipe('metadata_mlp')
frozen_base = default_recipe('frozen_bge')
RECIPES = {
    'metadata_pairwise': replace(
        metadata_base, ranking_loss='pairwise_logistic', hard_negatives_per_positive=0
    ),
    'metadata_lambda': replace(
        metadata_base, ranking_loss='lambda_ndcg', hard_negatives_per_positive=1
    ),
    'frozen_bge_pairwise': replace(
        frozen_base, ranking_loss='pairwise_logistic', hard_negatives_per_positive=0
    ),
    'frozen_bge_lambda': replace(
        frozen_base, ranking_loss='lambda_ndcg', hard_negatives_per_positive=1
    ),
}
assert {(recipe.approach, recipe.ranking_loss) for recipe in RECIPES.values()} == {
    ('metadata_mlp', 'pairwise_logistic'),
    ('metadata_mlp', 'lambda_ndcg'),
    ('frozen_bge', 'pairwise_logistic'),
    ('frozen_bge', 'lambda_ndcg'),
}
{name: asdict(recipe) for name, recipe in RECIPES.items()}

In [ ]:
results = {}
for model_name, recipe in RECIPES.items():
    print(f'\n=== {model_name} ===', flush=True)
    results[model_name] = run_neural_ranker_workflow(
        MODEL_DATA_ROOT,
        DATA_ROOT,
        OUTPUT_ROOT / model_name,
        approach=recipe.approach,
        embedding_root=EMBEDDING_ROOT,
        device=DEVICE,
        bootstrap_draws=BOOTSTRAP_DRAWS,
        training_mode=TRAINING_MODE,
        progress_every_stories=PROGRESS_EVERY,
        force_recompute=FORCE_RECOMPUTE,
        recipe=recipe,
    )
results

## Outputs

Each model directory contains separate `root/` and `all/` artifacts. Development-CV selection remains article-grouped and the sealed Paper 2 labels are used only for final evaluation. Rerunning the notebook reuses complete matching artifacts unless `COMMENTGAP_FORCE_RECOMPUTE=1` is set.